In [1]:
import pandas as pd

# =========================
# 1. Read files
# =========================
medals = pd.read_csv("data/olympic_medals.csv")
gdp = pd.read_csv("data/GDP.csv")

# =========================
# 2. Keep only needed columns
# =========================
medals = medals[
    [
        "discipline_title",
        "event_title",
        "event_gender",
        "medal_type",
        "participant_type",
        "country_name",
        "Area",
        "Year",
        "game_season",
        "Connect",
    ]
].copy()

gdp = gdp[
    [
        "Country Name",
        "Country Code",
        "Year",
        "Value",
        "AREA",
        "Connect",
        "Olympic-Year",
    ]
].copy()

gdp = gdp.rename(
    columns={
        "Country Name": "Country",
        "Value": "GDP",
        "AREA": "Area_gdp",
    }
)

# =========================
# 3. Clean data types
# =========================
medals["Year"] = pd.to_numeric(medals["Year"], errors="coerce")
gdp["Year"] = pd.to_numeric(gdp["Year"], errors="coerce")

medals = medals.dropna(subset=["Year", "country_name", "medal_type", "Connect"])
gdp = gdp.dropna(subset=["Year", "GDP", "Connect"])

medals["Year"] = medals["Year"].astype(int)
gdp["Year"] = gdp["Year"].astype(int)

medals = medals[medals["Year"] >= 1960]
gdp = gdp[gdp["Year"] >= 1960]

# =========================
# 4. Deduplicate medal records
# =========================
medals_unique = medals.drop_duplicates(
    subset=[
        "Connect",
        "discipline_title",
        "event_title",
        "event_gender",
        "medal_type",
        "country_name",
        "participant_type",
    ]
).copy()

# =========================
# 5. Aggregate medal counts
# =========================
# Medal count by Year × Country × MedalType
medal_count = (
    medals_unique.groupby(
        ["Year", "Connect", "country_name", "Area", "medal_type"],
        as_index=False
    )
    .size()
    .rename(
        columns={
            "country_name": "Country",
            "medal_type": "MedalType",
            "size": "MedalCount",
        }
    )
)

# =========================
# 6. Merge with GDP using Connect + Year
# =========================
merged = pd.merge(
    medal_count,
    gdp[["Year", "Connect", "Country", "GDP", "Area_gdp"]],
    on=["Year", "Connect"],
    how="inner",
    suffixes=("", "_gdp")
)

# =========================
# 7. Resolve country/area columns
# =========================
merged["Country"] = merged["Country"].fillna(merged["Country_gdp"]) if "Country_gdp" in merged.columns else merged["Country"]
merged["Area"] = merged["Area"].fillna(merged["Area_gdp"]) if "Area_gdp" in merged.columns else merged["Area"]

final_df = merged[["Year", "Country", "Area", "MedalType", "GDP", "MedalCount"]].copy()

final_df = final_df.dropna(subset=["GDP"])
final_df = final_df[final_df["GDP"] > 0]

final_df = final_df.sort_values(["Year", "Country", "MedalType"]).reset_index(drop=True)

# =========================
# 8. Save output
# =========================
final_df.to_csv("data/gdp_medals.csv", index=False)

print("Created gdp_medals.csv")
print(final_df.head(10))
print(f"Rows: {len(final_df)}")

Created gdp_medals.csv
   Year    Country           Area MedalType           GDP  MedalCount
0  1960  Australia        Oceania    BRONZE  1.860656e+10           6
1  1960  Australia        Oceania      GOLD  1.860656e+10           8
2  1960  Australia        Oceania    SILVER  1.860656e+10           8
3  1960    Austria         Europe    BRONZE  6.650134e+09           3
4  1960    Austria         Europe      GOLD  6.650134e+09           2
5  1960    Austria         Europe    SILVER  6.650134e+09           3
6  1960    Belgium         Europe    BRONZE  1.181062e+10           2
7  1960    Belgium         Europe    SILVER  1.181062e+10           2
8  1960     Canada  North America    BRONZE  4.056377e+10           1
9  1960     Canada  North America      GOLD  4.056377e+10           2
Rows: 2470
